# Лабораторная работа 1. Часть 1: нормализация и метрики
Все алгоритмы реализованы вручную на NumPy; sklearn используется только для проверки.

In [2]:
print("hello")

hello


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (mean_squared_error, mean_absolute_error,
                             r2_score, mean_absolute_percentage_error)

DATA_DIR = "."   # папка с train.csv и test.csv

## Подготовка данных
Убираем полностью пустые строки, аномальные значения цели (±1e7), точные дубли и строки с пропусками.
Дубли удаляем до любых разбиений, иначе одинаковые строки попадут и в обучение, и в проверку.

In [ ]:
raw = pd.read_csv(f"{DATA_DIR}/train.csv")
ok = raw["RiskScore"].notna() & (raw["RiskScore"].abs() < 1e6)
data = raw[ok].drop_duplicates().dropna().reset_index(drop=True)
print(f"было {len(raw)}, осталось {len(data)}")

num_cols = [c for c in data.select_dtypes("number").columns if c != "RiskScore"]
X_demo = data[num_cols].to_numpy(dtype=float)
y_demo = data["RiskScore"].to_numpy(dtype=float)
X_demo.shape

## 1. Нормализация
* **z-score:** `z = (x − μ) / σ` (σ с делителем n, как в sklearn);
* **min-max:** `x' = (x − min) / (max − min)`.

Параметры считаются только по обучающей выборке (`fit`), а затем применяются к любым данным (`transform`).
Для константного столбца знаменатель заменяется на 1, чтобы не делить на ноль.

In [ ]:
class ZScore:
    """Стандартизация: (x - mean) / std."""

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        self.mean_ = X.mean(axis=0)
        std = X.std(axis=0)                      # ddof=0
        self.std_ = np.where(std == 0, 1.0, std)
        return self

    def transform(self, X):
        return (np.asarray(X, dtype=float) - self.mean_) / self.std_

    def fit_transform(self, X):
        return self.fit(X).transform(X)

    def inverse_transform(self, Z):
        return np.asarray(Z, dtype=float) * self.std_ + self.mean_


class MinMax:
    """Приведение к диапазону [0, 1]: (x - min) / (max - min)."""

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        self.min_ = X.min(axis=0)
        span = X.max(axis=0) - self.min_
        self.span_ = np.where(span == 0, 1.0, span)
        return self

    def transform(self, X):
        return (np.asarray(X, dtype=float) - self.min_) / self.span_

    def fit_transform(self, X):
        return self.fit(X).transform(X)

    def inverse_transform(self, Z):
        return np.asarray(Z, dtype=float) * self.span_ + self.min_

In [ ]:
# проверка: обучаем на 80% строк, применяем к оставшимся 20% и сравниваем со sklearn
rng = np.random.RandomState(0)
perm = rng.permutation(len(X_demo))
tr, va = perm[:int(0.8 * len(perm))], perm[int(0.8 * len(perm)):]

for own, ref in [(ZScore(), StandardScaler()), (MinMax(), MinMaxScaler())]:
    own.fit(X_demo[tr]); ref.fit(X_demo[tr])
    d_tr = np.abs(own.transform(X_demo[tr]) - ref.transform(X_demo[tr])).max()
    d_va = np.abs(own.transform(X_demo[va]) - ref.transform(X_demo[va])).max()
    back = np.abs(own.inverse_transform(own.transform(X_demo[va])) - X_demo[va]).max()
    print(f"{type(own).__name__:7s} | макс. разница со sklearn: train {d_tr:.2e}, valid {d_va:.2e} | обратное преобразование: {back:.2e}")
    assert d_tr < 1e-8 and d_va < 1e-8

z = ZScore().fit_transform(X_demo[tr])
m = MinMax().fit_transform(X_demo[tr])
print("\nz-score: среднее ~", np.abs(z.mean(0)).max().round(12), "| std ~", z.std(0).round(6).min(), "-", z.std(0).round(6).max())
print("min-max: min =", m.min(), "| max =", m.max())

# константный столбец не ломает преобразование
const = np.column_stack([np.ones(5), np.arange(5.0)])
print("\nконстантный столбец:\n", ZScore().fit_transform(const).T, "\n", MinMax().fit_transform(const).T)

**Вывод.** Оба преобразования полностью совпадают со sklearn (разница на уровне 1e-16).
z-score нужен для градиентного спуска и регуляризации (признаки с разным масштабом иначе будут штрафоваться неодинаково),
min-max удобен, когда нужны значения в фиксированном диапазоне. Параметры на тестовой выборке не пересчитываются.

## 2. Метрики
Пусть `e_i = y_i − ŷ_i`.

* **MSE** = mean(e²) — сильнее штрафует большие ошибки; основная метрика соревнования.
* **MAE** = mean(|e|) — в единицах `RiskScore`.
* **R²** = 1 − Σe² / Σ(y − ȳ)² — доля объяснённой дисперсии.
* **MAPE** = mean(|e| / max(|y|, ε)), как в sklearn; возвращается доля (×100 даёт проценты).

In [ ]:
def _prep(y, pred):
    y = np.asarray(y, dtype=float).ravel()
    pred = np.asarray(pred, dtype=float).ravel()
    if y.shape != pred.shape:
        raise ValueError("y и pred должны иметь одинаковую длину")
    return y, pred


def mse(y, pred):
    y, pred = _prep(y, pred)
    return float(np.mean((y - pred) ** 2))


def mae(y, pred):
    y, pred = _prep(y, pred)
    return float(np.mean(np.abs(y - pred)))


def r2(y, pred):
    y, pred = _prep(y, pred)
    ss_res = np.sum((y - pred) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    if ss_tot == 0:                       # константная цель: как в sklearn
        return 1.0 if ss_res == 0 else 0.0
    return float(1.0 - ss_res / ss_tot)


def mape(y, pred):
    y, pred = _prep(y, pred)
    return float(np.mean(np.abs(y - pred) / np.maximum(np.abs(y), np.finfo(np.float64).eps)))

In [ ]:
# предсказания реальной модели (sklearn OLS) на отложенной части + искусственные случаи
from sklearn.linear_model import LinearRegression
model = LinearRegression().fit(ZScore().fit_transform(X_demo[tr]), y_demo[tr])
zs = ZScore().fit(X_demo[tr])
pred_real = model.predict(zs.transform(X_demo[va]))

cases = {
    "реальная модель": (y_demo[va], pred_real),
    "случайный шум": (rng.normal(50, 15, 1000), rng.normal(50, 15, 1000)),
    "идеальный прогноз": (y_demo[va], y_demo[va].copy()),
    "прогноз = среднее": (y_demo[va], np.full(len(va), y_demo[va].mean())),
}
rows = []
for name, (yt, yp) in cases.items():
    for metric, own, ref in [("MSE", mse, mean_squared_error), ("MAE", mae, mean_absolute_error),
                             ("R²", r2, r2_score), ("MAPE", mape, mean_absolute_percentage_error)]:
        a, b = own(yt, yp), ref(yt, yp)
        assert np.isclose(a, b, rtol=1e-10, atol=1e-12), (name, metric, a, b)
        rows.append((name, metric, a, b, abs(a - b)))
report = pd.DataFrame(rows, columns=["случай", "метрика", "своя", "sklearn", "|разница|"])
report.pivot(index="случай", columns="метрика", values="своя").round(5)

In [ ]:
report.round(8)

In [ ]:
# граничные случаи
print("R², константная цель, идеально:", r2([3, 3, 3], [3, 3, 3]), "| sklearn:", r2_score([3, 3, 3], [3, 3, 3]))
print("R², константная цель, с ошибкой:", r2([3, 3, 3], [2, 3, 4]), "| sklearn:", r2_score([3, 3, 3], [2, 3, 4]))
print("MAPE при y = 0:", mape([0.0, 1.0], [1.0, 1.0]), "| sklearn:", mean_absolute_percentage_error([0.0, 1.0], [1.0, 1.0]))

**Вывод.** Все четыре метрики совпадают со sklearn с точностью до ошибок округления на реальных и искусственных данных,
включая граничные случаи. Для итоговой оценки используется MSE, остальные метрики — для самопроверки.